Eddy-current problem expressed through a vector potential $A$:
\begin{align*}
\nabla\times(\nu\nabla\times\mathbf{A}) + j\omega\sigma \mathbf{A} = \mathbf{J}
\end{align*}
with
\begin{align*}
\mathbf{A}\times\mathbf{n} = \mathbf{0}
\end{align*}
on the far boundary. The magnetic permability $\mu_0 = 4\pi \cdot 10^{-7}$ H/m of vaccuum.

\begin{align*}
\text{Magnetic field} \; \; \; \mathbf{B} = \nabla\times\mathbf{A}
\end{align*}

This is a combination of two of Maxwell's Equations, Ampère's Law and Faraday's Law of Induction expressed though $\mathbf{A}$.

In [4]:
from ngsolve import *
from netgen.read_gmsh import ReadGmsh
from ngsolve.webgui import Draw
from netgen.csg import *
import math
import pyvista as pv
import numpy as np
from ngsolve.krylovspace import GMRes

# Import geometries
mesh = ReadGmsh('meshes/coil_box.msh')

for i in range(1, 3):
    # print(i)
    mesh.SetMaterial(i, f'{i}')

for i in range(1, 13):
    # print(i)
    mesh.SetBCName(i-1, f'{i}')

mesh = Mesh(mesh)

mesh.ngmesh.Save("meshes/coil_box.vol")

In [2]:
mesh.ne, mesh.nv, mesh.GetMaterials(), mesh.GetBoundaries()

(69629,
 12090,
 ('1', '2'),
 ('1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12'))

In [5]:
# Define material parameters

f = 123.5e3  # Frequency in Hz
I_coil = 2191.9 # A
mu0 = 4*math.pi*1e-7
omega = 2*math.pi*f # f Hz excitation

sigma = {"1": 0.0, "2": 5.998e7}  # Electric conductivity [S/m]
mu_r = {"1": 1.0, "2": 1.0} # Relative permeability [-]

mu_cf = mu0 * CoefficientFunction([mu_r.get(mat, 1.0) for mat in mesh.GetMaterials()])
sigma_cf = CoefficientFunction([sigma.get(mat, 0.0) for mat in mesh.GetMaterials()])

In [6]:
from ngsolve import *
from netgen.csg import unit_cube

crosssection = Integrate(1, mesh, definedon=mesh.Boundaries("8"))

print(f"Cross section: {crosssection}")

# 1. Define Spaces (Complex for Harmonic)
# HCurl for Vector Potential A, NumberSpace for the current constraint
fes_A = HCurl(mesh, order=1, complex=True)
fes_lam = NumberSpace(mesh, complex=True)
fes = fes_A * fes_lam

(A, lam), (v, mu) = fes.TnT()

# 2. Physical Parameters
f = 123.5e3  # Frequency in Hz
omega = 2*math.pi*f
mu_rev = 1.0           # Relative permeability

# 3. Bilinear Form (Harmonic Maxwell)
# Equation: curl(1/mu * curl A) + j*omega*sigma*A = J
a = BilinearForm(fes)
a += (1/mu_cf * curl(A) * curl(v)) * dx + 1e-6*A*v*dx
a += (1j * omega * sigma_cf * A * v) * dx

# Enforce the Total Current constraint on boundary "inlet"
# This couples the potential to the global current I0
a += lam * (v.Trace() * specialcf.normal(3)) * ds(definedon=mesh.Boundaries("8"))
a += mu * (A.Trace() * specialcf.normal(3)) * ds(definedon=mesh.Boundaries("8"))
a.Assemble()

# 4. Linear Form (The prescribed Current I0)
f = LinearForm(fes)
f += (I_coil / crosssection) * mu * ds(definedon=mesh.Boundaries("8"))
f.Assemble()

# pre = preconditioners.Local(a)
# pre = preconditioners.BDDC(a)
# A.vec.data = GMRes(a.mat, f.vec, pre=pre, maxsteps=5, printrates=True, tol= 1e-9)


# pre = preconditioners.BDDC(a)

# with TaskManager():
#     solvers.BVP(bf=a, lf=f, gf=A, pre=pre, \
#                 solver=solvers.CGSolver, solver_flags={"plotrates": True, "tol" : 1e-12})

Cross section: 7.0710678118656e-05


In [ ]:
pre = Preconditioner(a, type="bddc", inverse="sparsecholesky")
solvers.BVP(bf=a, lf=f, gf=A, pre=pre, \
                solver=solvers.CGSolver, solver_flags={"plotrates": True, "tol" : 1e-1})

In [4]:
# Check if the preconditioner itself is producing NaNs
pre = preconditioners.Local(a)
test_vec = f.vec.CreateVector()
test_vec.data = pre.mat * f.vec
print("Is Preconditioner OK?:", not np.isnan(np.sum(test_vec.FV().NumPy())))

# Check the diagonal of the matrix
import numpy as np
diag = a.mat.AsVector()
if np.any(np.abs(diag) == 0):
    print("Warning: Zero found on diagonal. 'Local' preconditioner will fail.")

Is Preconditioner OK?: False


In [ ]:
# Check if the preconditioner itself is producing NaNs
pre = preconditioners.BDDC(a)
a.Assemble()
test_vec = f.vec.CreateVector()
test_vec.data = pre.mat * f.vec
print("Is Preconditioner OK?:", not np.isnan(np.sum(test_vec.FV().NumPy())))

In [ ]:
pre = Preconditioner(a, "bddc")
a.Assemble()
gfu = GridFunction(fes)

gfu.vec.data = GMRes(a.mat, f.vec, pre=pre, maxsteps=10, printrates=True, tol= 1e-9)

In [11]:
res = f.vec.FV().NumPy()
print(res.shape)

print(np.max(res))
print(np.min(res))


vals = a.mat.AsVector()

print(f"Max value in matrix: {np.max(vals)}")
print(f"Min value in matrix: {np.min(vals)}")


is_nan = np.isnan(res)
print(is_nan)

print((is_nan == True).nonzero())


is_nan = np.isnan(vals)
print(is_nan)

print((is_nan == True).nonzero())


(668275,)
(2191.9+0j)
0j
Max value in matrix: (3288669727.5469866+0j)
Min value in matrix: (-1397498871.2496872+0j)
[False False False ... False False False]
(array([], dtype=int64),)
[False False False ... False False False]
(array([], dtype=int64),)


In [ ]:
crosssection = Integrate(1, mesh, definedon=mesh.Boundaries("8"))

# fes = HCurl(mesh, order=1, complex=True, dirichlet="VacuumSurface", gradientdomains="Tile|Block|Pipe")
fes = HCurl(mesh, order=1, complex=True, dirichlet="VacuumSurface", nograds = False)
print ("HCurl dofs:", fes.ndof)
u,v = fes.TnT()

a = BilinearForm(fes, symmetric=True, condense=True)
a += 1/mu_cf*curl(u)*curl(v)*dx+1e-6/mu_cf*u*v*dx
a += 1j*omega*sigma_cf * u*v*dx

pre = preconditioners.BDDC(a)

f = LinearForm(fes)

f += (I_coil / crosssection) * v * ds("8")

A = GridFunction(fes)

# with TaskManager():
#     solvers.BVP(bf=a, lf=f, gf=A, pre=pre, \
#                 solver=solvers.CGSolver, solver_flags={"plotrates": True, "tol" : 1e-12})

a.Assemble()
f.Assemble()
pre = preconditioners.Local(a)
A.vec.data = GMRes(a.mat, f.vec, pre=pre, maxsteps=100, printrates=True, tol= 1e-9)

HCurl dofs: 974082
GMRes iteration 1, residual = 40330.05692646058     
GMRes iteration 2, residual = 15122.881124603537     
GMRes iteration 3, residual = 10015.287740012493     
GMRes iteration 4, residual = 6367.644954505306     
GMRes iteration 5, residual = 3692.5645401110487     
GMRes iteration 6, residual = 2153.389736254386     
GMRes iteration 7, residual = 1316.1071041052232     
GMRes iteration 8, residual = 786.2231456227265     
GMRes iteration 9, residual = 454.3864835867998     
GMRes iteration 10, residual = 261.9095370561868     
GMRes iteration 11, residual = 152.20907003956177     
GMRes iteration 12, residual = 87.3895914171568     
GMRes iteration 13, residual = 49.923892138242856     
GMRes iteration 14, residual = 28.550037511783195     
GMRes iteration 15, residual = 16.689521095793197     
GMRes iteration 16, residual = 9.842130024630096     
GMRes iteration 17, residual = 5.7234458459943784     
GMRes iteration 18, residual = 3.297200109957543     
GMRes iter

In [14]:
B_cf = curl(A)
J_cf = -1j*omega*sigma_cf * A

E = -1j*omega * A
EE = sigma_cf * E * Conj(E)
q_cf = 0.5 * EE.real

sol = gfphi.vec.FV().NumPy()
res = pv.read('meshes/ERMES_2.msh')
res["J_coil"] = sol

fes_B = VectorH1(mesh, order=3)
B = GridFunction(fes_B)
B.Set(B_cf.real)

# fes_q = H1(mesh, order=1)
fes_q =H1(mesh, order=1, dim=1)
q = GridFunction(fes_q)
q.Set(q_cf)


fes_J = VectorH1(mesh, order=3)
J = GridFunction(fes_J)
J.Set(J_cf.real)


In [15]:

points = res.points

B_sol = np.zeros((points.shape[0], 3))

for i in range(points.shape[0]):
    point = mesh(points[i, 0], points[i, 1], points[i, 2])
    B_sol[i, :] = B(point)

res["B"] = B_sol


J_sol = np.zeros((points.shape[0], 3))

for i in range(points.shape[0]):
    point = mesh(points[i, 0], points[i, 1], points[i, 2])
    J_sol[i, :] = J(point)

res["J"] = J_sol

q_sol = np.zeros((points.shape[0], 1))

for i in range(points.shape[0]):
    point = mesh(points[i, 0], points[i, 1], points[i, 2])
    q_sol[i, :] = q(point)

res["q"] = q_sol

res.save('output/result_HIVE_code_aster.vtu')

In [7]:
# Save mesh + field + material ID

mat_id = GridFunction(L2(mesh))
mat_mapping = {name: i+1 for i, name in enumerate(mesh.GetMaterials())}

mat_cf = CoefficientFunction([mat_mapping.get(mat, 0) for mat in mesh.GetMaterials()])
mat_id.Set(mat_cf)

vtk = VTKOutput(ma=mesh,
                coefs=[q, mat_id],
                names=["q", "MaterialID"],
                filename="output/result_HIVE_code_aster2",
                subdivision=0)
vtk.Do()

'output/result_HIVE_code_aster2'